In [ ]:
from __future__ import annotations

import warnings

import numpy as np
import pandas as pd
from ase.visualize import view
from pymatgen.analysis.adsorption import AdsorbateSiteFinder
from pymatgen.core import Lattice, Molecule, Structure
from pymatgen.core.surface import SlabGenerator

from matcalc import AdsorptionCalc, PESCalculator, SurfaceCalc
from matcalc.utils import to_ase_atoms

warnings.filterwarnings("ignore", category=UserWarning, module="matgl")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="spglib")

/var/folders/m_/lthyt70x33l8vt_ydzdy7_900000gn/T/ipykernel_31015/4085868734.py:8: DeprecationWarning: This module has been moved to the pymatgen.core.adsorption module. This stub will be removed v2027.1. 
  from pymatgen.analysis.adsorption import AdsorbateSiteFinder


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# We use the M3GNet potential but it is certainly not a good
# choice as it is not trained for adsorption systems or the most
# accurate for materials available.

calc = PESCalculator.load_universal("r2SCAN")

In this notebook, we will calculate the minimum adsorption energy of CO on Pt using matcalc and a provided PESCalculator. We will do this in two steps:

1. Determine the lowest surface energy facet using SurfaceCalc
2. Determine the lowest adsorption energy and site using AdsorptionCalc

### 1. Surface Energy

In [ ]:
surf_calc = SurfaceCalc(
    calculator=calc,
    relax_bulk=True,
    relax_slab=True,
    fmax=0.01,
    optimizer="BFGS",
    max_steps=200,
    relax_calc_kwargs={},
)

In [ ]:
%%time
# Build BCC Pt conventional unit cell
bulk = Structure(
    Lattice.cubic(3.924),
    ["Pt"]*4,
    [[0,0,0],[0,0.5,0.5],[0.5,0,0.5],[0.5,0.5,0]],
)

# First calculate slab energies for 110 surface, if there are
# multiple terminations aka shifts, there will be multiple slabs returned
miller_index = (1, 1, 0)
slab_results = surf_calc.calc_slabs(
    bulk=bulk,
    miller_index=miller_index,
    min_slab_size=10.0,
    min_vacuum_size=15.0,
    symmetrize=True,
    inplane_supercell=(2, 2),
    slab_gen_kwargs={
        "center_slab": True,
        "max_normal_search": 1,
    },
    get_slabs_kwargs={
        "tol": 0.1,
    },
)

CPU times: user 1.25 s, sys: 379 ms, total: 1.63 s
Wall time: 856 ms


In [ ]:
for slab_result in slab_results:
    for k in slab_result:
        if k not in ["slab", "final_bulk", "final_slab"]:
            print(f"{k}: {slab_result[k]}")
    print("-"*50)

bulk_energy_per_atom: -51.46866226196289
miller_index: (1, 1, 0)
slab_energy: -1636.8594970703125
surface_energy: 0.11638747539046918
--------------------------------------------------


We will now calculate the surface energies of all facets up to a a miller index of 2 to determine the lowest energy site

In [ ]:
%%time
# Build BCC Pt conventional unit cell
bulk = Structure(
    Lattice.cubic(3.924),
    ["Pt"]*4,
    [[0,0,0],[0,0.5,0.5],[0.5,0,0.5],[0.5,0.5,0]],
)

# You can also use pymatgen's generate_all_slabs to get all unique Miller indices
# up to a certain value, here we get all unique Miller indices up to 2
miller_indices = [
    (1, 0, 0),
    (1, 1, 0),
    (1, 1, 1),
    (2, 0, 0),
    (2, 1, 0),
    (2, 1, 1),
    (2, 2, 1),
    (2, 2, 0),
]

slab_results = []
for miller_index in miller_indices:
    slab_results += surf_calc.calc_slabs(
        bulk=bulk,
        miller_index=miller_index,
        min_slab_size=10.0,
        min_vacuum_size=15.0,
        symmetrize=True,
        inplane_supercell=(2, 2),
        slab_gen_kwargs={
            "center_slab": True,
            "max_normal_search": 1,
        },
        get_slabs_kwargs={
            "tol": 0.1,
        },
    )

CPU times: user 28.5 s, sys: 7.78 s, total: 36.3 s
Wall time: 14.5 s


In [ ]:
# Some post-processing
df = pd.DataFrame()
for slab_result in slab_results:
    row = {
        "surface_energy": [slab_result["surface_energy"]],
        "miller_index": [slab_result["miller_index"]],
        "shift": [slab_result["slab"].shift],
    }
    df = pd.concat([df, pd.DataFrame(row)], ignore_index=True)

print(df.sort_values("surface_energy").reset_index(drop=True))

   surface_energy miller_index     shift
0        0.093631    (1, 1, 1)  0.166667
1        0.106498    (2, 2, 1)  0.050000
2        0.110739    (2, 1, 1)  0.062500
3        0.116387    (1, 1, 0)  0.125000
4        0.116387    (2, 2, 0)  0.125000
5        0.118145    (1, 0, 0)  0.250000
6        0.118145    (2, 0, 0)  0.250000
7        0.121168    (2, 1, 0)  0.083333


We see that the Pt(111) surface is lowest energy, so we proceed to calculate adsorption on this surface. Interestingly, M3GNet gets the lowest energy facet correct.

### 2. Adsorption on Pt(111) surface

We start by calculating the adsorption energy when we have the adsorbate, slab and adslab at hand. This method is parallelized in high-throughput calculations.

The structure inputs are quite flexible, they can be ASE or Pymatgen objects. Furthermore you can choose whether or not to relax the adsorbate and/or slab. If you choose not to, you can directly provide the "adsorbate_energy" or "slab_energy_per_atom" but you should still provide the structures. The slab can have a different area but must have the same number of layers, facet and any constraints on the motion of atoms.

Note that it is generally recommended to use experimental or high fidelity values for the adsorbate energies since DFT is not great for molecules. In particular, the O2 molecule triplet state is notoriously challenging fo most DFT functionals. In many cases, oxygen atom/molecule energies are obtained from experiment or linear combinations of reactions. See https://fair-chem.github.io/catalysts/examples_tutorials/OCP-introduction.html. To make matters worse, there is no foundational model known to get both atoms/molecules and materials correct anyway so referencing to a cheap DFT calculation is preferable.

In [ ]:
# # Using ASE objects
# slab = build.fcc111("Pt", size=(3, 3, 5), vacuum=15.0)
# # Fix bottom two layers regardles of whether the slab is centered
# c = FixAtoms(indices=[
#     atom.index for atom in slab if atom.position[2] < \
#         np.min(slab.positions, axis=0)[2] + 4.0]
#     )
# slab.set_constraint(c)
# adsorbate = build.molecule("CO")
# adslab = slab.copy()
# build.add_adsorbate(adslab, adsorbate, height=1.8, position="ontop")

# # We can visualize the structures, especially checking
# # that the adsorption site, size and constraints make sense
# view([slab, adsorbate, adslab])

In [ ]:
# Using pymatgen objects
bulk = Structure(
    Lattice.cubic(3.924),
    ["Pt"]*4,
    [
        [0, 0, 0],
        [0, 0.5, 0.5],
        [0.5, 0, 0.5],
        [0.5, 0.5, 0],
    ],
)
slabgen = SlabGenerator(
    initial_structure=bulk,
    miller_index=(1, 1, 1),
    min_slab_size=10.0,
    min_vacuum_size=15.0,
    center_slab=True,
)
slab = slabgen.get_slabs()[0]
slab.make_supercell((3, 3, 1))

# Apply constraints
minz = np.min(slab.cart_coords, axis=0)[2]
for site in slab:
    if site.coords[2] < minz + 4:
        site.properties["selective_dynamics"] = np.array(
            [False] * 3
        )
    else:
        site.properties["selective_dynamics"] = np.array(
            [True] * 3
        )

adsorbate = Molecule("HHO", [[0, 0, 0], [0, 0, 0.74], [0.76, 0, -0.24]])

adslab = slab.copy()
asf = AdsorbateSiteFinder(slab, height=0.4)
adsite = asf.find_adsorption_sites()["ontop"][0]
adslab = asf.add_adsorbate(adsorbate, adsite)

#Visualize structures
# view([
#     to_ase_atoms(a) for a in [slab, adsorbate, adslab]
# ])

In [ ]:
ad_calc = AdsorptionCalc(
    calculator=calc,
    relax_slab=True,
    relax_bulk=True,
    relax_adsorbate=False,
    fmax=0.05, #Not a very converged fmax!
    optimizer="BFGS",
    max_steps=200,
    relax_calc_kwargs={},
)

In [ ]:
results = ad_calc.calc(
    structure={
        "slab": slab,
        "adslab": adslab,
        "adsorbate": adsorbate,
        # "adsorbate_energy": -14.666958808898926, # Optionally provide adsorbate energy
    }
)

      Step     Time          Energy          fmax
BFGS:    0 01:57:00    -9230.325195       13.779482
BFGS:    1 01:57:00    -9231.247070        1.537150
BFGS:    2 01:57:00    -9231.329102        1.961520
BFGS:    3 01:57:01    -9231.938477        3.121528
BFGS:    4 01:57:01    -9233.020508        9.794456
BFGS:    5 01:57:01    -9232.635742       18.053513
BFGS:    6 01:57:01    -9233.940430        6.166558
BFGS:    7 01:57:01    -9234.399414        1.875677
BFGS:    8 01:57:01    -9234.500000        0.875189
BFGS:    9 01:57:02    -9234.539062        0.859807
BFGS:   10 01:57:02    -9234.686523        0.811346
BFGS:   11 01:57:02    -9234.714844        0.756453
BFGS:   12 01:57:02    -9234.782227        0.479142
BFGS:   13 01:57:02    -9234.803711        0.454342
BFGS:   14 01:57:02    -9234.818359        0.361679
BFGS:   15 01:57:02    -9234.829102        0.544761
BFGS:   16 01:57:03    -9234.844727        0.667252
BFGS:   17 01:57:03    -9234.861328        0.601838
BFGS:   18 01:

In [ ]:
for key in results:
    if key not in ["slab", "adslab", "adsorbate", "final_adslab", "final_adsorbate", "final_slab"]:
        print(f"{key}: {results[key]}")

adsorbate_energy: -10.204769134521484
slab_energy: -9218.828125
slab_energy_per_atom: -51.21571180555556
adslab_energy: -9235.197265625
adsorption_energy: -6.164371490478516


Let's now determine the lowest energy site by going through all the sites and calculating adsorption energies. We will however start by doing a dryrun since this calculation is a bit slow and you should check each of the generated slabs individually to make sure they are sensible!

In [ ]:
ac = AdsorptionCalc(
    calculator=calc,
    relax_slab=False,
    relax_bulk=False,
    relax_adsorbate=False,
    fmax=0.1,
    optimizer="BFGS",
    max_steps=200,
    relax_calc_kwargs={},
)

In [ ]:
bulk = Structure(
    Lattice.cubic(3.924),
    ["Pt"]*4,
    [
        [0, 0, 0],
        [0, 0.5, 0.5],
        [0.5, 0, 0.5],
        [0.5, 0.5, 0],
    ],
)
adsorbate = Molecule("CO", [[0, 0, 0], [0, 0, 1.13]])

In [ ]:
%%time
results = ac.calc_adslabs(
    adsorbate=adsorbate,
    adsorbate_energy=1.0,
    bulk=bulk,
    miller_index=(1,1,1),
    min_slab_size=10.0,
    min_vacuum_size=10.0,
    inplane_supercell=(2, 2),
    slab_gen_kwargs={},
    get_slabs_kwargs={},
    mi_vec=None,
    # adsorption_sites="all",
    adsorption_sites={"site1": [(0,0,13.5)], "site2": [(2,2,13.5), (0,0,13)]},  #Can specify custom sites
    height=0.7,
    fixed_height=4.0,
    find_adsorption_sites_args={},
    dry_run=True, # Avoids doing any calculations, just generates structures
)

CPU times: user 66.6 ms, sys: 37.7 ms, total: 104 ms
Wall time: 56.1 ms


In [ ]:
adslabs = [to_ase_atoms(res["adslab"]) for res in results]
# view(adslabs)

<Popen: returncode: None args: ['/Users/shyue/repos/matcalc/.venv/bin/python...>

In [ ]:
%%time
results = ac.calc_adslabs(
    adsorbate=adsorbate,
    # adsorbate_energy=1.0, # Optionally provide adsorbate energy to use
    bulk=bulk,
    miller_index=(1,1,1),
    min_slab_size=10.0,
    min_vacuum_size=10.0,
    inplane_supercell=(2, 2),
    slab_gen_kwargs={},
    get_slabs_kwargs={},
    adsorption_sites="all",
    height=0.7,
    fixed_height=4.0,
    find_adsorption_sites_args={},
    dry_run=False,
)

      Step     Time          Energy          fmax
BFGS:    0 01:57:15    -1248.024414        4.225470
BFGS:    1 01:57:15    -1248.078735        5.398405
BFGS:    2 01:57:15    -1248.295654        1.701866
BFGS:    3 01:57:15    -1248.337158        0.597343
BFGS:    4 01:57:15    -1248.339966        0.218663
BFGS:    5 01:57:15    -1248.345947        0.519445
BFGS:    6 01:57:15    -1248.355225        0.758936
BFGS:    7 01:57:15    -1248.365479        0.684990
BFGS:    8 01:57:15    -1248.378296        0.339720
BFGS:    9 01:57:15    -1248.393066        0.317820
BFGS:   10 01:57:15    -1248.407715        0.514820
BFGS:   11 01:57:15    -1248.417114        0.394179
BFGS:   12 01:57:15    -1248.420532        0.157572
BFGS:   13 01:57:15    -1248.422363        0.131905
BFGS:   14 01:57:16    -1248.424561        0.301977
BFGS:   15 01:57:16    -1248.426880        0.331349
BFGS:   16 01:57:16    -1248.429321        0.209725
BFGS:   17 01:57:16    -1248.431763        0.114352
BFGS:   18 01:

In [ ]:
df = pd.DataFrame()
for result in results:
    row = {
        "adsorption_energy": [result["adsorption_energy"]],
        "adsorption_site": [result["adsorption_site"]],
        "adsorption_site_index": [result["adsorption_site_index"]],
        "miller_index": [result["miller_index"]],
        "shift": [result["shift"]],
    }
    df = pd.concat([df, pd.DataFrame(row)], ignore_index=True)
df.sort_values("adsorption_energy")

,adsorption_energy,adsorption_site,adsorption_site_index,miller_index,shift
0,-2.371881,ontop,0,"(1, 1, 1)",0.166667
3,-2.255182,hollow,1,"(1, 1, 1)",0.166667
2,-2.248468,hollow,0,"(1, 1, 1)",0.166667
1,-2.173761,bridge,0,"(1, 1, 1)",0.166667


From the below reference the PBE adsorption energies are:

- Top: -1.80eV
- Bridge: -1.87eV
- Hollow: -1.88eV

So we're about 0.2eV off which isn't bad considering we used the M3GNet model with very loose convergence criteria. The hollow site is indeed the most stable.

Reference: https://pubs.acs.org/doi/full/10.1021/acscatal.8b00214